In [ ]:
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

from model_geokan_gamma import GeoKANGamma


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def load_regression_data(test_size=0.2, val_size=0.25, seed=42):
    # Example regression benchmark: Diabetes
    data = load_diabetes()
    X = data.data.astype(np.float32)                   # shape [N, 10]
    y = data.target.astype(np.float32).reshape(-1, 1)  # shape [N, 1]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=seed
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=val_size, random_state=seed
    )

    scaler_x = StandardScaler()
    X_train = scaler_x.fit_transform(X_train)
    X_val = scaler_x.transform(X_val)
    X_test = scaler_x.transform(X_test)

    # Optional but often helpful for regression
    scaler_y = StandardScaler()
    y_train = scaler_y.fit_transform(y_train)
    y_val = scaler_y.transform(y_val)
    y_test = scaler_y.transform(y_test)

    return (
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.float32),
        torch.tensor(X_val, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.float32),
        torch.tensor(X_test, dtype=torch.float32),
        torch.tensor(y_test, dtype=torch.float32),
        scaler_y,
    )


def train_model(
    model,
    X_train,
    y_train,
    X_val,
    y_val,
    epochs=400,
    lr=1e-3,
    weight_decay=1e-5,
    patience=50,
    device="cpu",
):
    model = model.to(device)
    X_train, y_train = X_train.to(device), y_train.to(device)
    X_val, y_val = X_val.to(device), y_val.to(device)

    criterion = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_state = copy.deepcopy(model.state_dict())
    best_val_loss = float("inf")
    bad_epochs = 0

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()

        pred = model(X_train)
        loss = criterion(pred, y_train)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(X_val)
            val_loss = criterion(val_pred, y_val).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if epoch % 100 == 0 or epoch == 1:
            print(
                f"Epoch {epoch:03d} | "
                f"train_mse={loss.item():.6f} | "
                f"val_mse={val_loss:.6f}"
            )

        if bad_epochs >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    model.load_state_dict(best_state)
    return model


def evaluate_model(model, X_test, y_test, scaler_y, device="cpu"):
    model.eval()
    X_test, y_test = X_test.to(device), y_test.to(device)

    with torch.no_grad():
        pred = model(X_test)

    y_true = y_test.cpu().numpy()
    y_pred = pred.cpu().numpy()

    # invert scaling to report real metrics
    y_true_orig = scaler_y.inverse_transform(y_true)
    y_pred_orig = scaler_y.inverse_transform(y_pred)

    mse = mean_squared_error(y_true_orig, y_pred_orig)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true_orig, y_pred_orig)

    print("\nTest Metrics")
    print("MSE :", mse)
    print("RMSE:", rmse)
    print("R2  :", r2)


def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"

    X_train, y_train, X_val, y_val, X_test, y_test, scaler_y = load_regression_data()

    # Diabetes: 10 input features, 1 output
    model = GeoKANGamma(
        in_dim=10,
        out_dim=1,
        width=12,
        depth=10,
        K=12,
    )

    print("Trainable params:", count_params(model))

    model = train_model(
        model,
        X_train,
        y_train,
        X_val,
        y_val,
        epochs=401,
        lr=2e-3,
        weight_decay=1e-2,
        patience=60,
        device=device,
    )

    evaluate_model(model, X_test, y_test, scaler_y, device=device)


if __name__ == "__main__":
    main()

Trainable params: 7567
Epoch 001 | train_mse=1.283577 | val_mse=0.919247
Epoch 100 | train_mse=0.595484 | val_mse=0.487918
Early stopping at epoch 158

Test Metrics
MSE : 3253.7437
RMSE: 57.041595
R2  : 0.385871946811676
